In [1]:
from torch_geometric.nn import RGCNConv, FastRGCNConv, global_mean_pool,global_add_pool, GINEConv, GATv2Conv, PNAConv
from sklearn.model_selection import train_test_split
from torch_geometric.loader import DataLoader
import torch.nn.functional as F
import torch.nn as nn
from pathlib import Path
import pandas as pd
import numpy as np
import random
import torch
import os 

In [2]:
ROOT_DIR = Path().cwd().parent
models_dir = ROOT_DIR / "models"
dataset_path = ROOT_DIR / "mutag-hetero"

# check if path exist 
if not models_dir.exists():
    raise FileNotFoundError(f"Models directory path {models_dir} does not exist.")


def set_seed(seed=42):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


set_seed(42)

In [3]:
class GINE(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_edge_types,
        num_classes,
    ):
        super().__init__()

        # Node feature encoder
        self.node_encoder = nn.Linear(in_channels, hidden_channels)

        # Bond type embedding
        self.edge_embedding = nn.Embedding(
            num_edge_types,
            hidden_channels
        )

        # GINE layer 1
        nn1 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )

        # GINE layer 2
        nn2 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )

        # GINE layer 3
        nn3 = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )

        self.conv1 = GINEConv(nn1, edge_dim=hidden_channels)
        self.conv2 = GINEConv(nn2, edge_dim=hidden_channels)
        self.conv3 = GINEConv(nn3, edge_dim=hidden_channels)

        self.dropout = nn.Dropout(0.5)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_channels, num_classes),
        )

    def forward(self, x, edge_index, edge_type, batch):

        x = self.node_encoder(x.float())

        edge_attr = self.edge_embedding(edge_type)

        x = F.relu(self.conv1(x, edge_index, edge_attr))
        x = F.relu(self.conv2(x, edge_index, edge_attr))
        x = F.relu(self.conv3(x, edge_index, edge_attr))

        x = global_add_pool(x, batch)

        x = self.dropout(x)

        return self.classifier(x)

In [4]:
class RGCN(torch.nn.Module):
    def __init__(self, in_channels, num_relations, hidden_channels=64, num_classes=2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = FastRGCNConv(hidden_channels, hidden_channels, num_relations)
        self.lin = torch.nn.Linear(hidden_channels, num_classes)

    def forward(self, x, edge_index, edge_type, batch):
        x = F.relu(self.conv1(x, edge_index, edge_type))
        x = F.relu(self.conv2(x, edge_index, edge_type))
        x = global_mean_pool(x, batch)
        return self.lin(x)

In [5]:
class GATv2_Molecule(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_edge_types,
        num_classes,
        heads=4 # Multi-head attention
    ):
        super().__init__()
        
        # We divide hidden_channels by heads so the final output dimension matches
        self.conv_hidden = hidden_channels // heads

        # Node feature encoder
        self.node_encoder = nn.Linear(in_channels, hidden_channels)

        # Bond type embedding (must match the input size expected by GAT edge_attr)
        self.edge_embedding = nn.Embedding(num_edge_types, self.conv_hidden * heads)

        # GATv2 Layers
        # Note: GATv2Conv natively accepts edge_attr
        self.conv1 = GATv2Conv(
            hidden_channels, 
            self.conv_hidden, 
            heads=heads, 
            edge_dim=self.conv_hidden * heads,
            dropout=0.2 # Internal attention dropout
        )
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = GATv2Conv(
            hidden_channels, 
            self.conv_hidden, 
            heads=heads, 
            edge_dim=self.conv_hidden * heads,
            dropout=0.2
        )
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.conv3 = GATv2Conv(
            hidden_channels, 
            self.conv_hidden, 
            heads=heads, 
            edge_dim=self.conv_hidden * heads,
            dropout=0.2
        )
        self.bn3 = nn.BatchNorm1d(hidden_channels)

        self.dropout = nn.Dropout(0.5)

        # Classifier uses Jumping Knowledge (concat of all 3 layers)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 3, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_channels, num_classes),
        )

    def forward(self, x, edge_index, edge_type, batch):
        # 1. Embed nodes and edges
        x = self.node_encoder(x.float())
        edge_attr = self.edge_embedding(edge_type)

        # 2. Message Passing with Attention
        x1 = self.conv1(x, edge_index, edge_attr=edge_attr)
        x1 = F.relu(self.bn1(x1))
        
        x2 = self.conv2(x1, edge_index, edge_attr=edge_attr)
        x2 = F.relu(self.bn2(x2))
        
        x3 = self.conv3(x2, edge_index, edge_attr=edge_attr)
        x3 = F.relu(self.bn3(x3))
        
        # 3. Jumping Knowledge: Combine representations from all layers
        x_jk = torch.cat([x1, x2, x3], dim=-1)

        # 4. Global Pooling
        pool_x = global_add_pool(x_jk, batch)

        # 5. Final classification
        return self.classifier(pool_x)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import PNAConv, global_add_pool

class PNA_Molecule(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_edge_types,
        num_classes,
        deg # <-- REQUIRED FOR PNA
    ):
        super().__init__()

        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.edge_embedding = nn.Embedding(num_edge_types, hidden_channels)

        # Define the multiple aggregators and scalers that make PNA special
        aggregators = ['mean', 'min', 'max', 'std']
        scalers = ['identity', 'amplification', 'attenuation']

        # PNA Layers (Notice we pass the aggregators, scalers, and deg tensor)
        self.conv1 = PNAConv(in_channels=hidden_channels, out_channels=hidden_channels,
                             aggregators=aggregators, scalers=scalers, deg=deg,
                             edge_dim=hidden_channels, towers=1, pre_layers=1, post_layers=1)
        self.bn1 = nn.BatchNorm1d(hidden_channels)

        self.conv2 = PNAConv(in_channels=hidden_channels, out_channels=hidden_channels,
                             aggregators=aggregators, scalers=scalers, deg=deg,
                             edge_dim=hidden_channels, towers=1, pre_layers=1, post_layers=1)
        self.bn2 = nn.BatchNorm1d(hidden_channels)

        self.conv3 = PNAConv(in_channels=hidden_channels, out_channels=hidden_channels,
                             aggregators=aggregators, scalers=scalers, deg=deg,
                             edge_dim=hidden_channels, towers=1, pre_layers=1, post_layers=1)
        self.bn3 = nn.BatchNorm1d(hidden_channels)

        self.dropout = nn.Dropout(0.5)

        # Classifier using Jumping Knowledge (Concat of 3 layers)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 3, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_channels, num_classes),
        )

    def forward(self, x, edge_index, edge_type, batch):
        x = self.node_encoder(x.float())
        edge_attr = self.edge_embedding(edge_type)

        x1 = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr=edge_attr)))
        x2 = F.relu(self.bn2(self.conv2(x1, edge_index, edge_attr=edge_attr)))
        x3 = F.relu(self.bn3(self.conv3(x2, edge_index, edge_attr=edge_attr)))
        
        x_jk = torch.cat([x1, x2, x3], dim=-1)
        pool_x = global_add_pool(x_jk, batch)
        
        return self.classifier(pool_x)

In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_type, data.batch)
        loss = criterion(out, data.y.squeeze())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(loader.dataset)


def evaluate(model, loader, device):
    model.eval()
    correct = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.edge_type, data.batch)
            pred = out.argmax(dim=1)
            correct += (pred == data.y.squeeze()).sum().item()
    return correct / len(loader.dataset)

In [8]:
BATCH_SIZE = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device("cpu")
print(f"[INFO] Training on device: {device}")

print("[INFO] Loading processed dataset...")
pyg_dataset = torch.load(dataset_path / "pyg_dataset.pt", weights_only=False)    
train_df = pd.read_csv(dataset_path / "trainingSet.tsv", sep="\t")
test_df = pd.read_csv(dataset_path / "testSet.tsv", sep="\t")


print("[INFO] Mapping training and testing molecule IDs to dataset indices...")
id_to_idx = {data.molecule_id: i for i, data in enumerate(pyg_dataset)}
  
train_ids = sorted(train_df["bond"].apply(lambda x: x.split("#")[-1]))
test_ids = sorted(test_df["bond"].apply(lambda x: x.split("#")[-1]))

train_idx = [id_to_idx[mol_id] for mol_id in train_ids if mol_id in id_to_idx]
test_idx  = [id_to_idx[mol_id] for mol_id in test_ids if mol_id in id_to_idx]
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42)

print("[INFO] Constructing train, test and validation DataLoader")
train_loader = DataLoader([pyg_dataset[i] for i in train_idx], batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader([pyg_dataset[i] for i in val_idx], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader([pyg_dataset[i] for i in test_idx], batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

  # Dynamically extract dimensions from the loaded data
in_channels = pyg_dataset[0].x.size(1)

# Extract total number of unique edge types across the dataset
num_relations = int(max(data.edge_type.max().item() for data in pyg_dataset) + 1)

[INFO] Training on device: cuda
[INFO] Loading processed dataset...
[INFO] Mapping training and testing molecule IDs to dataset indices...
[INFO] Constructing train, test and validation DataLoader


In [9]:
# # RGCN
# BATCH_SIZE = 32
# HIDDEN_CHANNELS = 64
# EPOCHS = 50
# LR = 0.001
# model = RGCN(
# in_channels=in_channels,
# num_relations=num_relations,
#     hidden_channels=HIDDEN_CHANNELS,
# num_classes=2
# ).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
# criterion = torch.nn.CrossEntropyLoss()
# print("[INFO] Starting training RGCN...")

# for epoch in range(1, EPOCHS + 1):
#     loss = train_epoch(model, train_loader, optimizer, criterion, device)
#     train_acc = evaluate(model, train_loader, device)
#     val_acc = evaluate(model, val_loader, device)
        
#     if epoch % 5 == 0 or epoch == 1:
#         print(f'Epoch {epoch:02d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

# test_acc = evaluate(model, test_loader, device)
# print(f'[INFO] Final Test Accuracy: {test_acc:.4f}')

In [ ]:
# BATCH_SIZE = 32
# HIDDEN_CHANNELS = 64
# EPOCHS = 100
# LR = 0.001

# # Number of unique bond types
num_edge_types = num_relations

# model = GINE(
#     in_channels=in_channels,
#     hidden_channels=HIDDEN_CHANNELS,
#     num_edge_types=num_edge_types,
#     num_classes=2,
# ).to(device)

# optimizer = torch.optim.AdamW(
#     model.parameters(),
#     lr=LR,
#     weight_decay=5e-4,
# )

# criterion = torch.nn.CrossEntropyLoss()

# print("[INFO] Starting training GINE...")
# for epoch in range(1, EPOCHS + 1):

#     loss = train_epoch(
#         model,
#         train_loader,
#         optimizer,
#         criterion,
#         device,
#     )

#     train_acc = evaluate(model, train_loader, device)
#     val_acc = evaluate(model, val_loader, device)

#     if epoch % 5 == 0 or epoch == 1:
#         print(
#             f"Epoch {epoch:02d}, "
#             f"Loss: {loss:.4f}, "
#             f"Train Acc: {train_acc:.4f}, "
#             f"Val Acc: {val_acc:.4f}"
#         )

# test_acc = evaluate(model, test_loader, device)

# print(f"[INFO] Final Test Accuracy: {test_acc:.4f}")

In [ ]:
# only for PNA Model
from torch_geometric.utils import degree
print("[INFO] Computing degree distribution for PNA...")
max_degree = -1
for data in pyg_dataset:
    # edge_index[1] represents the target nodes (in-degree)
    d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
    max_degree = max(max_degree, int(d.max()))

# Create a histogram of degrees
deg = torch.zeros(max_degree + 1, dtype=torch.long)
for data in pyg_dataset:
    d = degree(data.edge_index[1], num_nodes=data.num_nodes, dtype=torch.long)
    deg += torch.bincount(d, minlength=deg.numel())

# Convert to float for PNA scaling
deg = deg.to(torch.float)

[INFO] Computing degree distribution for PNA...


In [12]:
import numpy as np
import torch
from sklearn.model_selection import KFold
from torch_geometric.loader import DataLoader

# --- 1. Configuration ---
BATCH_SIZE = 16
HIDDEN_CHANNELS = 16
EPOCHS = 50
LR = 0.001
K_FOLDS = 5

# Note: Ensure 'dataset' is a list of your PyG Data objects 
# or your full PyTorch Geometric dataset instance before this point.

# --- 2. Helper to get a fresh model ---
def get_fresh_model(name):

    if name == 'gine':
        return GINE(
            in_channels=in_channels,
            hidden_channels=HIDDEN_CHANNELS,
            num_edge_types=num_edge_types,
            num_classes=2,
        ).to(device)
    
    elif name == 'rgcn':
        return RGCN(
        in_channels=in_channels,
        num_relations=num_relations,
        hidden_channels=HIDDEN_CHANNELS,
        num_classes=2
        ).to(device)
    
    elif name == 'gatv2':
        return GATv2_Molecule(
            in_channels=in_channels,
            hidden_channels=HIDDEN_CHANNELS,
            num_edge_types=num_edge_types, # Ensure this matches your dataset!
            num_classes=2,
            heads=4
        ).to(device)

    elif name == 'pna':
        return PNA_Molecule(
            in_channels=in_channels,
            hidden_channels=HIDDEN_CHANNELS,
            num_edge_types=num_edge_types,
            num_classes=2,
            deg=deg # <-- Pass the degree tensor here!
        ).to(device)
    

kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_test_accs = []

print(f"[INFO] Starting {K_FOLDS}-Fold Cross Validation...")

for fold, (train_idx, test_idx) in enumerate(kfold.split(pyg_dataset)):
    print(f"\n--- Fold {fold + 1}/{K_FOLDS} ---")
    
    train_subset = [pyg_dataset[i] for i in train_idx]
    test_subset = [pyg_dataset[i] for i in test_idx]
    
    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False)
    
    # INITIALIZE FRESH MODEL & OPTIMIZER FOR EACH FOLD
    model = get_fresh_model('pna')
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=5e-4,
    )
    criterion = torch.nn.CrossEntropyLoss()
    
    for epoch in range(1, EPOCHS + 1):
        loss = train_epoch(
            model, 
            train_loader, 
            optimizer, 
            criterion, 
            device
        )
        
        # Printing less frequently to keep the console clean
        if epoch % 10 == 0 or epoch == EPOCHS:
            train_acc = evaluate(model, train_loader, device)
            print(f"  Epoch {epoch:03d} | Loss: {loss:.4f} | Train Acc: {train_acc:.4f}")
    
    # Evaluate on this Fold's Test Set
    test_acc = evaluate(model, test_loader, device)
    print(f"> Fold {fold + 1} Test Accuracy: {test_acc:.4f}")
    
    fold_test_accs.append(test_acc)

# --- 5. Final Results ---
mean_acc = np.mean(fold_test_accs)
std_acc = np.std(fold_test_accs)

print("\n=========================================")
print(f"[INFO] Final 10-Fold CV Results")
print(f"Accuracy: {mean_acc * 100:.2f}% ± {std_acc * 100:.2f}%")
print("=========================================")

[INFO] Starting 5-Fold Cross Validation...

--- Fold 1/5 ---


/home/vaibhav/miniconda3/envs/xai/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/home/vaibhav/miniconda3/envs/xai/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


  Epoch 010 | Loss: 0.4933 | Train Acc: 0.8419
  Epoch 020 | Loss: 0.4133 | Train Acc: 0.8382
  Epoch 030 | Loss: 0.3534 | Train Acc: 0.9118
  Epoch 040 | Loss: 0.2958 | Train Acc: 0.9265
  Epoch 050 | Loss: 0.3306 | Train Acc: 0.9375
> Fold 1 Test Accuracy: 0.8088

--- Fold 2/5 ---
  Epoch 010 | Loss: 0.4557 | Train Acc: 0.8199
  Epoch 020 | Loss: 0.3858 | Train Acc: 0.7426
  Epoch 030 | Loss: 0.3144 | Train Acc: 0.8934
  Epoch 040 | Loss: 0.2997 | Train Acc: 0.8934
  Epoch 050 | Loss: 0.2678 | Train Acc: 0.9044
> Fold 2 Test Accuracy: 0.8529

--- Fold 3/5 ---
  Epoch 010 | Loss: 0.4356 | Train Acc: 0.8493
  Epoch 020 | Loss: 0.3711 | Train Acc: 0.8787
  Epoch 030 | Loss: 0.3192 | Train Acc: 0.8640
  Epoch 040 | Loss: 0.3570 | Train Acc: 0.8897
  Epoch 050 | Loss: 0.3500 | Train Acc: 0.8493
> Fold 3 Test Accuracy: 0.8088

--- Fold 4/5 ---
  Epoch 010 | Loss: 0.4297 | Train Acc: 0.8529
  Epoch 020 | Loss: 0.4166 | Train Acc: 0.8713
  Epoch 030 | Loss: 0.3742 | Train Acc: 0.9007
  Epoch

In [21]:
import torch
import matplotlib.pyplot as plt
from torch_geometric.explain import Explainer, GNNExplainer

print("\n[INFO] Starting Explainability Phase with GNNExplainer...")

model.eval()

explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type='model',
    node_mask_type='attributes', 
    edge_mask_type='object',     
    model_config=dict(
        mode='multiclass_classification',
        task_level='graph',
        return_type='raw', 
    ),
)

target_graph_idx = -1
for i, data in enumerate(pyg_dataset):
    data = data.to(device)
    batch = torch.zeros(data.x.size(0), dtype=torch.long).to(device)
    
    with torch.no_grad():
        out = model(data.x, data.edge_index, data.edge_type, batch)
        pred = out.argmax(dim=1).item()
        
    if data.y.item() == 1 and pred == 1:
        target_graph_idx = i
        target_data = data
        target_batch = batch
        break

if target_graph_idx == -1:
    print("Could not find a correctly predicted mutagenic graph.")
else:
    print(f"Explaining Graph Index: {target_graph_idx} (Molecule ID: {target_data.molecule_id})")

    explanation = explainer(
        target_data.x, 
        target_data.edge_index, 
        edge_type=target_data.edge_type, 
        batch=target_batch
    )

    # --------------------------------------------------------------
    # FIX ADDED HERE
    idx_to_atom = {}
    for d in pyg_dataset:
        indices = d.x.argmax(dim=1).tolist()
        atom_types = [atom["type"] for atom in d.atom_info]
        for idx, atom_type in zip(indices, atom_types):
            idx_to_atom[idx] = atom_type
    # --------------------------------------------------------------

    node_importance = explanation.node_mask.sum(dim=0) 
    top_features = torch.topk(node_importance, k=5)
    
    print("\n--- Top 5 Most Important RDF Atom Types ---")
    for rank, (idx, score) in enumerate(zip(top_features.indices, top_features.values)):
        idx_val = idx.item()
        atom_name = idx_to_atom.get(idx_val, f"Unknown-{idx_val}")
        print(f"{rank+1}. {atom_name} (Score: {score.item():.4f})")

    atom_labels = [atom["type"] for atom in target_data.atom_info]

    image_path = f'pna_explanation_{target_data.molecule_id}.png'
    explanation.visualize_graph(
        backend="networkx",
        node_labels=atom_labels,
        path=image_path
    )
    print(f"\n[INFO] Saved labeled subgraph visualization to '{image_path}'")


[INFO] Starting Explainability Phase with GNNExplainer...
Explaining Graph Index: 3 (Molecule ID: d101)

--- Top 5 Most Important RDF Atom Types ---
1. Carbon-10 (Score: 5.1998)
2. Carbon-16 (Score: 3.6356)
3. Hydrogen-3 (Score: 1.3189)
4. Chlorine-93 (Score: 1.1091)
5. Arsenic-101 (Score: 0.0000)

[INFO] Saved labeled subgraph visualization to 'pna_explanation_d101.png'


In [23]:
# 4. Extract Importance per Individual Atom (Node)
# By summing across dim=1 (features), we get a single score for each physical atom in the graph
atom_importance = explanation.node_mask.sum(dim=1)
    
print(f"\n--- Importance Scores for All Atoms in Molecule {target_data.molecule_id} ---")
    
# We will store them in a list so we can sort them from highest to lowest
scored_atoms = []
    
for node_idx in range(target_data.num_nodes):
    # We grab the exact string name directly from your saved metadata!
    atom_name = target_data.atom_info[node_idx]["type"]
    score = atom_importance[node_idx].item()
    scored_atoms.append((node_idx, atom_name, score))
        
# Sort the atoms so the most important ones print at the top
scored_atoms.sort(key=lambda x: x[2], reverse=True)
    
for node_idx, atom_name, score in scored_atoms:
    print(f"Atom {node_idx:02d} [{atom_name}]: {score:.4f}")


--- Importance Scores for All Atoms in Molecule d101 ---
Atom 20 [Carbon-16]: 0.9188
Atom 21 [Carbon-16]: 0.9149
Atom 00 [Carbon-10]: 0.9115
Atom 16 [Carbon-16]: 0.9093
Atom 11 [Carbon-10]: 0.9015
Atom 19 [Carbon-10]: 0.9009
Atom 15 [Carbon-16]: 0.8926
Atom 05 [Carbon-10]: 0.8398
Atom 18 [Carbon-10]: 0.8271
Atom 17 [Carbon-10]: 0.8191
Atom 01 [Hydrogen-3]: 0.7332
Atom 14 [Chlorine-93]: 0.1997
Atom 12 [Chlorine-93]: 0.1671
Atom 03 [Hydrogen-3]: 0.1624
Atom 07 [Chlorine-93]: 0.1568
Atom 08 [Chlorine-93]: 0.1548
Atom 06 [Chlorine-93]: 0.1526
Atom 09 [Hydrogen-3]: 0.1471
Atom 10 [Chlorine-93]: 0.1432
Atom 02 [Hydrogen-3]: 0.1387
Atom 04 [Hydrogen-3]: 0.1376
Atom 13 [Chlorine-93]: 0.1349
